<a href="https://colab.research.google.com/github/SeoYun-Y/thinfiler-credit-scoring/blob/track-b%2F03-final-handoff/notebooks/track-b/track_b_03_%EC%A0%84%EC%B2%98%EB%A6%AC%EC%99%84%EB%A3%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track B — 전처리 완료 데이터 전달용 노트북

담당: 서윤 | 선행 노트북: track_b_01_타겟변수_검증.ipynb, track_b_02_대안변수_선정.ipynb

본 노트북은 앞선 두 노트북에서 확정한 타겟변수와 46개 대안변수를 처음부터 재현하여, 모델링 담당자(김태헌)에게 전달할 최종 CSV 2개를 생성하는 것을 목적으로 한다. 탐색·시행착오 과정은 생략하고 확정된 결과만 담는다.

**최종 산출물**
- `track_b_features_train.csv` — 신용이력 보유군 285,890명, 46개 변수 + TARGET (학습·검증용)
- `track_b_features_apply.csv` — 씬파일러 44,110명, 46개 변수만 (정답 없이 적용 전용)

## 0. 데이터 구조 요약 — X, y, 씬파일러 정의 필드의 관계

본 노트북 산출물을 처음 접하는 사람을 위해, 두 CSV 파일 안에 무엇이 들어있고 무엇이 빠져있는지 먼저 명확히 한다.

**씬파일러 정의 필드(5개)는 두 CSV 어디에도 컬럼으로 존재하지 않는다.**

`PYE_C1M210000`, `PYE_C18233003~005`, `PYE_L10210000` 5개 필드는 신용이력 보유군과 씬파일러를 나누는 기준으로만 사용하고, 예측변수(X)에는 포함하지 않았다. 이는 정의에 사용한 필드를 예측에 다시 사용하면 순환논리(정의 기준과 예측 변수가 중복되어 결과가 왜곡되는 문제)가 발생하기 때문이다. 대신 이 정의를 이미 적용해 분리된 두 개의 파일을 산출물로 제공한다.

| 파일 | X(46개 변수) | y(TARGET) | 씬파일러 정의 필드 | 대상 |
|---|---|---|---|---|
| track_b_features_train.csv | 포함 | 포함 | 미포함 | 신용이력 보유군(정의 조건 불충족자) |
| track_b_features_apply.csv | 포함 | 미포함 | 미포함 | 씬파일러(정의 조건 충족자) |

즉 두 파일로 나뉘어 있다는 사실 자체가 씬파일러 정의를 적용한 결과이며, 별도의 정의 필드를 열어볼 필요 없이 '어느 파일에 속해 있는가'로 세그먼트를 구분할 수 있다.

**X(피처)와 y(타겟)의 구성**

- **X**: 두 CSV 공통으로 존재하는 46개 컬럼(CUST_ID, TARGET 제외 전부) — DAR, TOT_ASST, PYE_AL012G019 등 자산·부채·생활안정성·소비패턴·관심도·라이프스타일 등급으로 구성
- **y(TARGET)**: train.csv에만 존재하는 컬럼. `PYE_D1011000C > 0`(1년 내 연체 발생, 해제포함)을 0/1로 변환한 것. 1=연체 있었음(양성, 12,088명), 0=연체 없었음(273,802명)
- apply.csv에 TARGET이 없는 이유: 씬파일러는 카드·대출 자체가 없어 연체라는 사건이 성립하기 어렵고, `PYE_D1011000C` 값이 있어도 신뢰할 수 있는 정답으로 쓸 수 없음이 track_b_01에서 이미 검증됨. 따라서 씬파일러에는 학습된 모델을 적용하여 점수만 산출하고, 정답과 대조하는 검증은 수행하지 않는다.

## 1. 환경 설정

드라이브를 마운트하고 원본 데이터 경로를 지정한다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base_path = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
scan_path = f'{base_path}/통신카드CB_씬파일러.csv'

Mounted at /content/drive


## 2. 최종 확정 변수 목록 정의

track_b_02에서 738개 필드를 IV·다중공선성·순환논리·씬파일러 유효성 검증을 거쳐 확정한 46개 변수를 세 그룹으로 나누어 정의한다.

- **핵심 변수 38개**: 자산·부채, 생활안정성(이력변경), 주거, 소비패턴, 관심도 등
- **등급 관련 파생 5개**: 문자형 변수(PET/APP/GOLF/TRAVEL_GD)에서 파생된 WOE 인코딩 4개 + 등급산정불가 플래그(grade_unavailable) 1개
- **결측 처리 플래그 3개**: DAR, FST_CAR_ELPS, FST_HOUS_BUY_ELPS의 결측 여부

In [2]:
# 핵심 변수 43개 (원본 42개 + 문자형 4개를 woe/플래그 5개로 대체)
final_features = [
    'DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC',
    'R3M_ITRT_ENT_BROADCAST', 'AGE', 'OWN_HOUS_CNT', 'FAM_OWN_HOUS_CNT',
    'PYE_CAR_OWN', 'JB_TP', 'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS',
    'FAM_OWN_LIV_YN', 'RET_SIL', 'CPYT_CONV_AMT_RT_was_missing',
    'YOY_R3M_FOOD_AMT_RTC', 'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC',
    'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC', 'PYE_AL012G011', 'SHP_CNT',
    'R3M_ITRT_SHOP_JIKGU', 'YOY_R3M_MART_AMT_RTC', 'B1Y_MOB_OS',
    'CPYT_FOOD_AMT_RT_was_missing', 'QOQ_R3M_MART_AMT_RTC', 'HOME_ADM',
    'YOY_R3M_HOUSEHOLD_AMT_RTC', 'R3M_ITRT_ENT_WEBTOON', 'R12M_MED_AMT',
    'R3M_ITRT_FIN_INSUR', 'YOY_R3M_ITRT_COMM_VOIP_CS', 'YOY_R3M_CONV_AMT_RTC',
    'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC'
]

# 문자형에서 파생된 변수 (등급 불가 플래그 + WOE 인코딩 4개)
grade_features = ['grade_unavailable', 'PET_GD_woe', 'APP_GD_woe', 'GOLF_GD_woe', 'TRAVEL_GD_woe']

# 결측 처리 플래그 3개
missing_flags = ['DAR_was_missing', 'FST_CAR_ELPS_was_missing', 'FST_HOUS_BUY_ELPS_was_missing']

print(f"핵심 변수: {len(final_features)}개")
print(f"등급 관련 파생: {len(grade_features)}개")
print(f"결측 플래그: {len(missing_flags)}개")
print(f"총합: {len(final_features) + len(grade_features) + len(missing_flags)}개")

핵심 변수: 38개
등급 관련 파생: 5개
결측 플래그: 3개
총합: 46개


**결과**: 38 + 5 + 3 = 46개로 확인되었다.

## 3. 202103 원본 데이터 로드

202103(2020년말 기준) 파일에서 필요한 컬럼을 불러온다. 단, `_was_missing`으로 끝나는 파생 플래그(CPYT_CONV_AMT_RT_was_missing 등)는 원본 CSV에 존재하지 않는 계산된 컬럼이므로, 그 원본 소스 컬럼(CPYT_CONV_AMT_RT 등)을 대신 불러와 이후 단계에서 직접 생성한다. 씬파일러 정의 필드 5개와 문자형 원본 4개(PET_GD 등)도 함께 로드한다.

In [4]:
thin_def_cols = ['PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']
raw_grade_cols = ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']

# _was_missing으로 끝나는 파생 플래그는 원본 컬럼명으로 변환해서 별도 로드
derived_flag_cols = [c for c in final_features if c.endswith('_was_missing')]
raw_source_cols = [c.replace('_was_missing', '') for c in derived_flag_cols]

# read_csv로 바로 불러올 수 있는 컬럼만 추리기
direct_load_cols = [c for c in final_features if not c.endswith('_was_missing')]

load_cols = ['CUST_ID'] + direct_load_cols + raw_grade_cols + thin_def_cols + raw_source_cols
load_cols = list(dict.fromkeys(load_cols))  # 중복 제거

df_202103 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=load_cols)
print(df_202103.shape)
print(f"파생 플래그 원본 소스 컬럼: {raw_source_cols}")

(330000, 48)
파생 플래그 원본 소스 컬럼: ['CPYT_CONV_AMT_RT', 'CPYT_FOOD_AMT_RT']


**결과**: (330000, 48) — 정상적으로 로드되었다.

## 4. 파생 플래그 생성 및 결측 처리

CPYT_CONV_AMT_RT, CPYT_FOOD_AMT_RT의 결측 여부를 플래그로 만들고, DAR·FST_CAR_ELPS·FST_HOUS_BUY_ELPS는 결측을 0으로 채우면서 동시에 결측 여부 플래그를 생성한다. 이 세 변수의 결측은 각각 부채비율 계산 불가(총자산 0), 차량·주택 미구입을 의미하는 구조적 결측임이 track_b_02에서 이미 확인된 바 있다.

In [5]:
# 1. CPYT was_missing 플래그 생성
for col in ['CPYT_CONV_AMT_RT', 'CPYT_FOOD_AMT_RT']:
    df_202103[f'{col}_was_missing'] = df_202103[col].isnull().astype(int)

# 2. DAR, FST_CAR_ELPS, FST_HOUS_BUY_ELPS 결측 처리
for col in ['DAR', 'FST_CAR_ELPS', 'FST_HOUS_BUY_ELPS']:
    df_202103[f'{col}_was_missing'] = df_202103[col].isnull().astype(int)
    df_202103[col] = df_202103[col].fillna(0)

print("결측 처리 후 확인:")
print(df_202103[['DAR', 'FST_CAR_ELPS', 'FST_HOUS_BUY_ELPS']].isnull().sum())
print(f"\n생성된 플래그: {[c for c in df_202103.columns if c.endswith('_was_missing')]}")

결측 처리 후 확인:
DAR                  0
FST_CAR_ELPS         0
FST_HOUS_BUY_ELPS    0
dtype: int64

생성된 플래그: ['CPYT_CONV_AMT_RT_was_missing', 'CPYT_FOOD_AMT_RT_was_missing', 'DAR_was_missing', 'FST_CAR_ELPS_was_missing', 'FST_HOUS_BUY_ELPS_was_missing']


**결과**: 결측 0으로 처리되었고, 플래그 5개(CPYT_CONV, CPYT_FOOD, DAR, FST_CAR_ELPS, FST_HOUS_BUY_ELPS)가 정상 생성되었다.

## 5. 씬파일러 정의 및 타겟(TARGET) 병합

5개 필드가 전부 0인 경우를 씬파일러로 정의하여 신용이력 보유군/씬파일러로 분리한다. 202203 파일에서 PYE_D1011000C(1년내 발생 연체, 해제포함)를 불러와 TARGET(0/1)으로 변환하고 병합한다.

In [6]:
# 씬파일러 정의
thin_mask = (
    (df_202103['PYE_C1M210000']==0) & (df_202103['PYE_C18233003']==0) &
    (df_202103['PYE_C18233004']==0) & (df_202103['PYE_C18233005']==0) &
    (df_202103['PYE_L10210000']==0)
)
df_202103['segment'] = thin_mask.map({True: 'thin_filer', False: 'credit_holder'})

# y 로드 (202203의 PYE_D1011000C)
df_y = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=['CUST_ID', 'PYE_D1011000C'])
df_y['TARGET'] = (df_y['PYE_D1011000C'] > 0).astype(int)

df_full = df_202103.merge(df_y[['CUST_ID', 'TARGET']], on='CUST_ID', how='left')

credit_holders = df_full[df_full['segment']=='credit_holder'].copy()
thin_filers = df_full[df_full['segment']=='thin_filer'].copy()

print(f"신용이력 보유군: {len(credit_holders):,}명, 양성 {credit_holders['TARGET'].sum():,}명 ({credit_holders['TARGET'].mean()*100:.3f}%)")
print(f"씬파일러: {len(thin_filers):,}명")

신용이력 보유군: 285,890명, 양성 12,088명 (4.228%)
씬파일러: 44,110명


**결과**: 신용이력 보유군 285,890명(양성 12,088명, 4.228%), 씬파일러 44,110명 — track_b_01, track_b_02와 정확히 일치함을 확인하였다.

## 6. 문자형 변수 WOE 인코딩

PET_GD, APP_GD, GOLF_GD, TRAVEL_GD 4개 문자형 변수의 특수값('*')을 grade_unavailable 이진 플래그로 분리하고, 0~3 등급 부분만을 대상으로 WOE를 계산한다.

**중요한 원칙**: WOE 매핑값은 반드시 정답이 있는 신용이력 보유군에서 계산하고, 그 매핑을 씬파일러에도 동일하게 적용한다. 씬파일러는 정답이 없으므로 자체적으로 WOE를 계산하지 않고, 학습된 규칙을 그대로 가져다 쓰는 구조를 여기서도 일관되게 유지한다.

In [7]:
# grade_unavailable 플래그 생성 (양쪽 데이터프레임 모두)
credit_holders['grade_unavailable'] = (credit_holders['PET_GD'] == '*').astype(int)
thin_filers['grade_unavailable'] = (thin_filers['PET_GD'] == '*').astype(int)

def woe_encode_nonstar(df, feature, target):
    data = df[df[feature] != '*'][[feature, target]].copy()
    grouped = data.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()
    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)
    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    return grouped['woe'].to_dict()

# WOE는 신용이력보유군(정답 있는 집단) 기준으로 계산 -> 씬파일러에도 동일하게 적용
for col in ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']:
    woe_map = woe_encode_nonstar(credit_holders, col, 'TARGET')
    credit_holders[f'{col}_woe'] = credit_holders[col].map(woe_map).fillna(0)
    thin_filers[f'{col}_woe'] = thin_filers[col].map(woe_map).fillna(0)

print("WOE 인코딩 완료")
print(credit_holders[['PET_GD_woe','APP_GD_woe','GOLF_GD_woe','TRAVEL_GD_woe','grade_unavailable']].describe())

WOE 인코딩 완료
          PET_GD_woe     APP_GD_woe    GOLF_GD_woe  TRAVEL_GD_woe  \
count  285890.000000  285890.000000  285890.000000  285890.000000   
mean        0.006415       0.003282       0.000585       0.000006   
std         0.112935       0.081358       0.038987       0.017547   
min        -0.439396      -0.435158      -0.256245      -0.044438   
25%         0.044012       0.025843       0.000645      -0.002066   
50%         0.044012       0.025843       0.000645      -0.002066   
75%         0.044012       0.025843       0.000645      -0.002066   
max         0.044012       0.025843       0.271811       0.093821   

       grade_unavailable  
count      285890.000000  
mean            0.149414  
std             0.356497  
min             0.000000  
25%             0.000000  
50%             0.000000  
75%             0.000000  
max             1.000000  


**결과**: WOE 값 분포와 grade_unavailable 비율(14.94%)이 track_b_02와 일치함을 확인하였다.

## 7. 최종 변수 목록 검증

핵심 38개, 등급 파생 5개, 결측 플래그 3개를 통합하여 46개 최종 목록을 구성하고, 누락 여부·결측·문자형 잔존 여부를 확인한다.

In [8]:
final_all_features = final_features + grade_features + missing_flags

# 46개 검증
print(f"최종 변수 개수: {len(final_all_features)}개")

missing_in_df = [c for c in final_all_features if c not in credit_holders.columns]
print(f"credit_holders에 누락된 변수: {missing_in_df}")

# 결측 최종 확인
print(f"\ncredit_holders 결측 총합: {credit_holders[final_all_features].isnull().sum().sum()}")
print(f"thin_filers 결측 총합: {thin_filers[final_all_features].isnull().sum().sum()}")

# 문자형 남아있는지 확인
dtype_check = credit_holders[final_all_features].dtypes
print(f"\n문자형 남아있는 변수: {(dtype_check=='object').sum()}개")

최종 변수 개수: 46개
credit_holders에 누락된 변수: []

credit_holders 결측 총합: 1431291
thin_filers 결측 총합: 259875

문자형 남아있는 변수: 0개


**중간 결과**: 누락된 변수는 없었으나(46개 전부 존재), 결측이 credit_holders 1,431,291개·thin_filers 259,875개로 다수 발견되었다.

## 8. 잔여 결측 원인 확인

결측이 발견된 컬럼을 개별 확인한다.

In [9]:
missing_by_col = credit_holders[final_all_features].isnull().sum()
missing_by_col = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
print(missing_by_col)

YOY_R3M_M_CONV_AMT_RTC       247120
YOY_R3M_CONV_AMT_RTC         231399
QOQ_R3M_MART_AMT_RTC         230165
YOY_R3M_MART_AMT_RTC         218654
YOY_R3M_HOUSEHOLD_AMT_RTC    180770
YOY_R3M_FOOD_AMT_RTC         157534
YOY_R3M_SSM_AMT_RTC          147027
QOQ_TOT_ASST_RTC               9490
YOY_TOT_ASST_RTC               9132
dtype: int64


**원인**: 전부 QOQ_/YOY_ 접두사가 붙은 파생(변화율) 변수로 확인되었다. 이는 track_b_02에서 이미 검증한 패턴으로, 해당 소비·활동 자체가 없어 증감률을 계산할 대상이 없는 구조적 결측이다. track_b_02에서는 161개 파생변수 전체에 이 처리를 적용했으나, 본 노트북에서는 최종 46개만 재현하는 과정에서 해당 처리 단계가 누락되었던 것으로 확인되어 아래에서 보정한다.

## 9. 파생변수(QOQ_/YOY_) 결측 재처리

결측이 확인된 9개 파생변수를 0으로 채운다.

In [10]:
derived_missing_cols = ['YOY_R3M_M_CONV_AMT_RTC', 'YOY_R3M_CONV_AMT_RTC', 'QOQ_R3M_MART_AMT_RTC',
                          'YOY_R3M_MART_AMT_RTC', 'YOY_R3M_HOUSEHOLD_AMT_RTC', 'YOY_R3M_FOOD_AMT_RTC',
                          'YOY_R3M_SSM_AMT_RTC', 'QOQ_TOT_ASST_RTC', 'YOY_TOT_ASST_RTC']

for col in derived_missing_cols:
    credit_holders[col] = credit_holders[col].fillna(0)
    thin_filers[col] = thin_filers[col].fillna(0)

# 재검증
print(f"credit_holders 결측 총합: {credit_holders[final_all_features].isnull().sum().sum()}")
print(f"thin_filers 결측 총합: {thin_filers[final_all_features].isnull().sum().sum()}")

credit_holders 결측 총합: 0
thin_filers 결측 총합: 0


**결과**: credit_holders, thin_filers 양쪽 모두 결측 0으로 확인되어, 46개 변수가 완전히 정리되었다.

## 10. 최종 CSV 저장

신용이력 보유군은 학습용(46개 변수 + CUST_ID + TARGET), 씬파일러는 적용용(46개 변수 + CUST_ID, TARGET 제외)으로 분리하여 저장한다.

In [11]:
export_cols_train = ['CUST_ID'] + final_all_features + ['TARGET']
export_cols_apply = ['CUST_ID'] + final_all_features

df_train = credit_holders[export_cols_train]
df_apply = thin_filers[export_cols_apply]

df_train.to_csv(f'{base_path}/변수스캔결과/track_b_features_train.csv', index=False)
df_apply.to_csv(f'{base_path}/변수스캔결과/track_b_features_apply.csv', index=False)

print(f"학습용 저장 완료: {df_train.shape}")
print(f"적용용 저장 완료: {df_apply.shape}")
print(f"\n학습용 TARGET 분포:\n{df_train['TARGET'].value_counts()}")

학습용 저장 완료: (285890, 48)
적용용 저장 완료: (44110, 47)

학습용 TARGET 분포:
TARGET
0    273802
1     12088
Name: count, dtype: int64


**최종 결과**

| 파일 | 크기 | 내용 |
|---|---|---|
| track_b_features_train.csv | 285,890 × 48 | 신용이력 보유군, 46개 변수 + CUST_ID + TARGET |
| track_b_features_apply.csv | 44,110 × 47 | 씬파일러, 46개 변수 + CUST_ID |

TARGET 분포: 0(정상) 273,802명 / 1(연체) 12,088명(4.228%) — 기존 검증 결과와 일치함을 확인하였다.

---

## 전달 시 함께 공유할 사항

1. **타겟 정의**: PYE_D1011000C > 0(1년내 연체발생, 해제포함). X는 202103(2020년말), y는 202203(2021년 발생분) 시점으로 분리.
2. **씬파일러 적용 시 해석 주의 변수(10개)**: DAR, HIGHEND_CD2, RET_SIL, PYE_CAR_OWN, R6M_M_MART_AMT, SHP_CNT, OWN_HOUS_CNT, QOQ_R3M_MART_AMT_RTC, R6M_MART_AMT, PYE_AL012G011 — 신용이력 보유군 내 최빈값 비율 90% 이상으로, 씬파일러 세그먼트에서는 변별력이 제한적일 수 있음.
3. **스모크 테스트 결과**: 로지스틱회귀(StandardScaler 적용) 기준 AUC 0.7073, PR-AUC 0.1112.
4. **범위 제한 사항**: 시계열(분기별 궤적) 변수는 데이터 범위 한계로 이번 단계에서 제외하였으며, 추후 과거 시점 데이터 확보 시 확장을 검토할 수 있음.
5. CSV 파일은 GitHub에 커밋하지 않으며, 구글 드라이브 공유 폴더로만 전달함(팀 협업 규칙).